# Task 3: QLoRA Fine-tuning & Evaluation (Qwen2.5-Coder-7B)

Notebook huấn luyện và đánh giá mô hình **Qwen2.5-Coder-7B-Instruct** cho Task 3 (sinh phản hồi sư phạm tiếng Việt có kiểm soát mức độ):
- **Fine-tuning**: QLoRA 4-bit trên tập dữ liệu kết hợp (`mixed_balanced` gồm real-train và synthetic-train).
- **Inference & Evaluation**: Suy luận trên tập kiểm thử công khai (`public_test_release`), kiểm tra tuân thủ sư phạm và xuất file submission chuẩn format.

---
### Cấu hình Kaggle khuyến nghị:
1. **Accelerator**: Chọn **GPU T4 x2** hoặc **GPU P100**.
2. **Internet**: Bật **On** để tải packages và model weights.
3. **Dữ liệu**: Upload các file data (`generate-dataset.zip`, `sample_dataset.zip`, `public_test_release.zip`) vào Kaggle Input hoặc dùng Google Drive ID bên dưới.


In [ ]:
# ==============================================================================
# 1. CẤU HÌNH THỰC NGHIỆM
# ==============================================================================

# Git repository & branch
REPO = "github.com/tiendinhquang2104/llm-final-project.git"
BRANCH = "task-3"

# Google Drive ID dự phòng (nếu không upload qua Kaggle Input)
DRIVE_ID_PUBLIC_SAMPLE = "1gfMQ9LpCbg2AOwPfHnt7qe220J1AfLBj"
DRIVE_ID_SYNTHETIC = "1oESzEa-vwppDD-dDTmAVk7KIHqUsgowl"

# Chế độ thực thi:
# - "qwen_qlora": Huấn luyện QLoRA và chạy suy luận
# - "qwen_baseline": Chạy suy luận trực tiếp với mô hình gốc
# - "gemini": Chạy baseline với Gemini API
MODE = "qwen_qlora"

# Split đánh giá: "public" (328 mẫu Public Test) hoặc "real-val"
EVAL_SPLIT = "public"

# Tham số huấn luyện
EXPERIMENT = "mixed_balanced"
EPOCHS = 1
MAX_STEPS = None

WORK = "/kaggle/tmp"


In [ ]:
# ==============================================================================
# 2. KHỞI TẠO MÔI TRƯỜNG VÀ ĐỒNG BỘ CODE
# ==============================================================================

import os, sys, shutil, subprocess
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    def get_secret(k):
        try:
            return secrets.get_secret(k)
        except Exception:
            return None
except Exception:
    def get_secret(k):
        return os.environ.get(k)

gh_pat = get_secret("GH_PAT") or get_secret("GITHUB_TOKEN")
url = f"https://{gh_pat}@{REPO}" if gh_pat else f"https://{REPO}"

os.makedirs(WORK, exist_ok=True)
REPO_DIR = f"{WORK}/repo"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO} ({BRANCH})...")
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("Không thể clone repo, vui lòng kiểm tra nhánh hoặc quyền truy cập.") from None
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
else:
    print(f"Pulling latest changes from {BRANCH}...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "origin", BRANCH])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Nạp API keys nếu có trong Kaggle Secrets
env_vars = {}
for key in ["GEMINI_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"]:
    val = get_secret(key)
    if val:
        os.environ[key] = val
        env_vars[key] = val

if env_vars:
    with open(".env", "w", encoding="utf-8") as f:
        f.write("\n".join(f"{k}={v}" for k, v in env_vars.items()) + "\n")

print(f"Working directory: {os.getcwd()}")
print("Git commit:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())


In [ ]:
# ==============================================================================
# 3. CÀI ĐẶT THƯ VIỆN VÀ KIỂM TRA GPU
# ==============================================================================

!pip install -q gdown pyyaml pytest
!pip install -q "transformers>=4.51,<5" accelerate bitsandbytes peft trl datasets
if MODE in ("gemini", "openai"):
    !pip install -q google-genai openai

!nvidia-smi


In [ ]:
# ==============================================================================
# 4. CHUẨN BỊ DỮ LIỆU
# ==============================================================================

import os, glob, shutil, gdown

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/synth", exist_ok=True)
os.makedirs(f"{WORK}/download", exist_ok=True)

# 4.1. Sample dataset & Public test
print("=== [1/3] Chuẩn bị sample_dataset và public_test_release ===")
found_sample = glob.glob("/kaggle/input/**/sample_dataset", recursive=True)
found_public = glob.glob("/kaggle/input/**/public_test_release", recursive=True)
found_real_zips = glob.glob("/kaggle/input/**/sample_dataset*.zip", recursive=True) or glob.glob("/kaggle/input/**/*cham-diem*.zip", recursive=True)

if found_sample and not os.path.exists("data/raw/sample_dataset"):
    shutil.copytree(found_sample[0], "data/raw/sample_dataset")
    print(f"Loaded sample_dataset from {found_sample[0]}")

if found_public and not os.path.exists("data/raw/public_test_release"):
    shutil.copytree(found_public[0], "data/raw/public_test_release")
    print(f"Loaded public_test_release from {found_public[0]}")

if not os.path.exists("data/raw/sample_dataset") or not os.path.exists("data/raw/public_test_release"):
    zip_source = None
    if found_real_zips:
        zip_source = found_real_zips[0]
        print(f"Found archive: {zip_source}")
    else:
        drive_id = DRIVE_ID_PUBLIC_SAMPLE or get_secret("PUBLIC_TEST_DRIVE_ID") or get_secret("DRIVE_ID")
        if drive_id:
            zip_dest = f"{WORK}/download/public_and_sample.zip"
            print(f"Downloading from Google Drive (ID: {drive_id})...")
            gdown.download(id=drive_id, output=zip_dest, quiet=False)
            zip_source = zip_dest

    if zip_source and os.path.exists(zip_source):
        !python scripts/unpack_data.py {os.path.dirname(zip_source)} --dest data/raw
        for sub_zip in glob.glob("data/raw/*.zip"):
            !unzip -q -o {sub_zip} -d data/raw

!rm -rf data/raw/__MACOSX

# 4.2. Synthetic dataset
print("\n=== [2/3] Chuẩn bị synthetic dataset ===")
synth_dest = "data/synth/generate-dataset"
found_synth_dirs = [d for d in glob.glob("/kaggle/input/**/generate-dataset*", recursive=True) if os.path.isdir(d) and os.path.exists(os.path.join(d, "task3_feedback.json"))]
found_synth_zips = glob.glob("/kaggle/input/**/generate-dataset*.zip", recursive=True)

if found_synth_dirs and not os.path.exists(synth_dest):
    shutil.copytree(found_synth_dirs[0], synth_dest)
    print(f"Loaded generate-dataset from {found_synth_dirs[0]}")
elif found_synth_zips and not os.path.exists(synth_dest):
    print(f"Extracting synthetic dataset from {found_synth_zips[0]}")
    !unzip -q -o {found_synth_zips[0]} -d data/synth
else:
    drive_synth_id = DRIVE_ID_SYNTHETIC or get_secret("SYNTHETIC_DRIVE_ID")
    if drive_synth_id:
        synth_zip = f"{WORK}/download/generate_dataset.zip"
        print(f"Downloading synthetic dataset from Drive (ID: {drive_synth_id})...")
        gdown.download(id=drive_synth_id, output=synth_zip, quiet=False)
        !unzip -q -o {synth_zip} -d data/synth
    elif not os.path.exists(synth_dest):
        print("Note: Chưa tìm thấy generate-dataset. Có thể bổ sung qua '+ Add Data'.")

if os.path.exists(f"{synth_dest}/generate-dataset"):
    for item in os.listdir(f"{synth_dest}/generate-dataset"):
        shutil.move(f"{synth_dest}/generate-dataset/{item}", f"{synth_dest}/{item}")
    shutil.rmtree(f"{synth_dest}/generate-dataset")

!rm -rf data/synth/__MACOSX

# 4.3. Tiền xử lý dữ liệu Task 3
print("\n=== [3/3] Tiền xử lý dữ liệu Task 3 ===")
!python scripts/task3_prepare.py --filter-synthetic strict

print("\nPartitions in data/processed/task3:")
!ls -lh data/processed/task3


In [ ]:
# ==============================================================================
# 5. CẤU HÌNH THIẾT BỊ GPU
# ==============================================================================

import os, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gpu_count = torch.cuda.device_count()
print(f"Phát hiện {gpu_count} GPU khả dụng:")
for i in range(gpu_count):
    free_mem = torch.cuda.mem_get_info(i)[0] / (1024**3)
    total_mem = torch.cuda.mem_get_info(i)[1] / (1024**3)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | Free VRAM: {free_mem:.1f}/{total_mem:.1f} GiB")

if gpu_count > 1:
    print(f"Kích hoạt multi-GPU (device_map='auto') trên {gpu_count} GPU.")
else:
    print("Chạy trên single-GPU.")


In [ ]:
# ==============================================================================
# 6. HUẤN LUYỆN QLORA
# ==============================================================================

import os, subprocess, shutil
from pathlib import Path

adapter_dir = f"checkpoints/task3/{EXPERIMENT}"
working_adapter = "/kaggle/working/task3_lora_adapter"

if MODE == "qwen_qlora":
    print(f"Bắt đầu huấn luyện Qwen2.5-Coder-7B QLoRA (experiment={EXPERIMENT}, epochs={EPOCHS})...")

    train_cmd = [
        "python", "scripts/train_task3.py",
        "--epochs", str(EPOCHS),
        "--config", "configs/task3/qlora.yaml",
        "--experiment", EXPERIMENT,
        "--output-dir", adapter_dir
    ]
    if MAX_STEPS:
        train_cmd.extend(["--max-steps", str(MAX_STEPS)])

    subprocess.run(train_cmd, check=True)
    print(f"\nHuấn luyện hoàn tất. Adapter lưu tại: {adapter_dir}")

    # Sao lưu LoRA adapter sang /kaggle/working để tải về
    try:
        source_save = f"{adapter_dir}/adapter" if os.path.exists(f"{adapter_dir}/adapter") else adapter_dir
        shutil.copytree(source_save, working_adapter, dirs_exist_ok=True)
        print(f"Đã sao lưu LoRA adapter sang: {working_adapter}")
    except Exception as e:
        print(f"Cảnh báo khi sao lưu adapter: {e}")
else:
    print(f"Bỏ qua bước huấn luyện (MODE = '{MODE}').")


In [ ]:
# ==============================================================================
# 7. SUY LUẬN BATCH
# ==============================================================================

import os, json, subprocess, shutil
from pathlib import Path

output_file = f"outputs/task3/{MODE}_{EVAL_SPLIT}.predictions.json"
os.makedirs("outputs/task3", exist_ok=True)

# Chuẩn bị baseline taxonomy cho tập Public Test nếu chưa có
taxonomy_file = "outputs/task3/public_baseline_taxonomy.json"
if EVAL_SPLIT == "public":
    from scripts.build_task3_submission import build_default_taxonomy
    if not os.path.exists(taxonomy_file):
        build_default_taxonomy(Path("data/raw/public_test_release"), Path(taxonomy_file))

print(f"\nBắt đầu suy luận trên split: {EVAL_SPLIT}...")

infer_cmd = ["python", "scripts/run_task3.py", "--split", EVAL_SPLIT, "--output", output_file, "--context-limit", "8192"]

if EVAL_SPLIT == "public":
    infer_cmd.extend(["--taxonomy-source", "task2-json", "--taxonomy-json", taxonomy_file, "--allow-unverified-task2-export"])
else:
    infer_cmd.extend(["--taxonomy-source", "gold"])

if MODE == "gemini":
    infer_cmd.extend(["--config", "configs/task3/gemini.yaml"])
elif MODE == "openai":
    infer_cmd.extend(["--config", "configs/task3/openai.yaml"])
elif MODE == "qwen_baseline":
    infer_cmd.extend(["--config", "configs/task3/baseline.yaml"])
elif MODE == "qwen_qlora":
    working_adapter = "/kaggle/working/task3_lora_adapter"
    if os.path.exists(f"{working_adapter}/adapter_model.safetensors") or os.path.exists(f"{working_adapter}/adapter_model.bin"):
        target_adapter = working_adapter
    elif os.path.exists(f"{adapter_dir}/adapter"):
        target_adapter = f"{adapter_dir}/adapter"
    else:
        target_adapter = adapter_dir
    infer_cmd.extend(["--config", "configs/task3/baseline.yaml", "--adapter", target_adapter])

infer_cmd.append("--resume-progress")
print("Command:", " ".join(infer_cmd))

infer_env = os.environ.copy()
infer_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if "CUDA_VISIBLE_DEVICES" in infer_env:
    del infer_env["CUDA_VISIBLE_DEVICES"]

subprocess.run(infer_cmd, env=infer_env, check=True)
print(f"\nSuy luận hoàn tất. Kết quả lưu tại: {output_file}")


In [ ]:
# ==============================================================================
# 8. ĐÁNH GIÁ ĐỘ TUÂN THỦ VÀ XUẤT SUBMISSION
# ==============================================================================

import os, json, shutil, subprocess

traces_file = output_file.replace(".predictions.json", ".predictions.traces.json")
eval_file = output_file.replace(".predictions.json", ".evaluation.json")

# 1. Đo lường chỉ số tuân thủ và chất lượng phản hồi
eval_cmd = [
    "python", "scripts/evaluate_task3.py",
    "--data-dir", "data/processed/task3",
    "--split", EVAL_SPLIT,
    "--predictions", output_file,
    "--traces", traces_file,
    "--taxonomy-source", ("task2-json" if EVAL_SPLIT == "public" else "gold"),
    "--output", eval_file
]
subprocess.run(eval_cmd, check=True)

# 2. Xuất file submission ra /kaggle/working/
submission_dest = "/kaggle/working/task3_submission.json"
shutil.copy(output_file, submission_dest)
if os.path.exists(eval_file):
    shutil.copy(eval_file, f"/kaggle/working/{os.path.basename(eval_file)}")

print("\n--- Báo cáo đánh giá (Evaluation Summary) ---")
with open(eval_file, "r", encoding="utf-8") as f:
    eval_res = json.load(f)
    print(json.dumps(eval_res.get("summary", eval_res), indent=2, ensure_ascii=False))

# 3. Kiểm tra file submission
with open(submission_dest, "r", encoding="utf-8") as f:
    preds = json.load(f)

print(f"\nSubmission: {len(preds)} mẫu đã được lưu tại {submission_dest}")
print("3 mẫu đầu tiên:")
print(json.dumps(preds[:3], indent=2, ensure_ascii=False))

print("\nCác file trong /kaggle/working/:")
!ls -lh /kaggle/working
